# TwinLiteNet+ 로컬 파인튜닝 (Windows + AMD RX 9070 XT / ROCm)

Colab 노트북(`finetune_twinlitenetplus.ipynb`)의 로컬 버전. Colab 무료 GPU 할당량이
반복적으로 소진돼서(PROGRESS.md §2.12/§3) `medium` config 학습을 집 PC(9070 XT,
시스템램 64GB)에서 직접 돌리기 위한 것 — Google Drive 마운트 대신 로컬 폴더를 쓰고,
ROCm PyTorch 설치 셀이 추가된 것 말고는 데이터 파이프라인/패치/학습 로직은 Colab판과
동일하다.

**사전 준비 (이 노트북 실행 전에 꼭 확인)**
1. **AMD 그래픽 드라이버**: Adrenalin **26.2.2** 이상 설치돼 있을 것(ROCm 7.2.1 PyTorch
   Windows 릴리즈 요구사항).
2. **Python 3.12**로 이 커널을 띄울 것 — ROCm Windows wheel이 cp312 전용으로 배포됨
   (다른 버전이면 아래 설치 셀이 실패함). `python --version`으로 먼저 확인.
3. **`BASE_DIR`**(아래 cell 2)를 실제 이 프로젝트를 둘 로컬 경로로 바꿀 것.
4. 아래 파일들을 `BASE_DIR` 바로 밑에 미리 복사해둘 것(Mac에서 만든 것 그대로 옮기면 됨):
   - `pseudo_dataset_v2.zip` (1068장, da=bootstrap_v2 모델/ll=YOLOPv2+skeleton, §2.12)
   - `pretrained/medium.pth` (없으면 아래 셀이 자동 다운로드 시도)

**Git**: `git`이 PATH에 있어야 저장소 clone 셀이 동작함(Git for Windows 설치 필요,
https://git-scm.com/download/win). 없으면 GitHub에서 zip으로 받아 `REPO_DIR`에 직접
풀어놓고 이 셀은 건너뛰어도 됨.

## 0. ROCm PyTorch 설치 (최초 1회만 — 이미 설치했으면 건너뛰고 1번으로)

공식 AMD 문서(rocm.docs.amd.com, Radeon용 Windows 네이티브 설치) 기준. **2단계**로
나뉜다: 먼저 ROCm SDK 자체를 설치하고, 그 다음 ROCm 빌드 torch/torchvision/torchaudio를
설치한다. 버전(7.2.1)은 문서 확인 시점 기준 — 실행 전에
https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html
에서 최신 버전 번호로 URL이 바뀌었는지 한 번 확인할 것.

In [ ]:
# --- 1단계: ROCm SDK (버전 번호는 위 공식 문서에서 최신인지 확인 후 실행) ---
!pip install --no-cache-dir https://repo.radeon.com/rocm/windows/rocm-rel-7.2.1/rocm_sdk_core-7.2.1-py3-none-win_amd64.whl https://repo.radeon.com/rocm/windows/rocm-rel-7.2.1/rocm_sdk_devel-7.2.1-py3-none-win_amd64.whl https://repo.radeon.com/rocm/windows/rocm-rel-7.2.1/rocm_sdk_libraries_custom-7.2.1-py3-none-win_amd64.whl https://repo.radeon.com/rocm/windows/rocm-rel-7.2.1/rocm-7.2.1.tar.gz

In [ ]:
# --- 2단계: ROCm 빌드 PyTorch (cp312 전용 wheel) ---
!pip install --no-cache-dir "https://repo.radeon.com/rocm/windows/rocm-rel-7.2.1/torch-2.9.1%2Brocm7.2.1-cp312-cp312-win_amd64.whl" "https://repo.radeon.com/rocm/windows/rocm-rel-7.2.1/torchaudio-2.9.1%2Brocm7.2.1-cp312-cp312-win_amd64.whl" "https://repo.radeon.com/rocm/windows/rocm-rel-7.2.1/torchvision-0.24.1%2Brocm7.2.1-cp312-cp312-win_amd64.whl"

In [ ]:
# GPU 인식 확인 - RX 9070 XT가 device name으로 떠야 정상. ROCm은 HIP이 CUDA API를
# 그대로 흉내내므로(torch.cuda.*), 아래 학습 스크립트(finetune.py)는 CUDA 전용
# 코드 수정 없이 그대로 재사용 가능함.
import torch
print('torch', torch.__version__)
print('cuda(=ROCm/HIP) available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 1. 환경 설정 (로컬 경로)

In [ ]:
import os

# ★ 실행 전에 반드시 실제 경로로 바꿀 것 ★
BASE_DIR = r'C:\Users\YOUR_NAME\umk_twinlite_local'

WORK_DIR = os.path.join(BASE_DIR, 'work')
REPO_DIR = os.path.join(WORK_DIR, 'TwinLiteNetPlus')
DATA_DIR = os.path.join(WORK_DIR, 'bdd100k')  # BDD100K.py가 '../bdd100k' 상대경로로 참조 - 이름 그대로 유지

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(BASE_DIR, exist_ok=True)
print('BASE_DIR:', BASE_DIR)
print('이 폴더 바로 밑에 pseudo_dataset_v2.zip / pretrained/medium.pth를 미리 넣어둘 것')

In [ ]:
%cd {WORK_DIR}
!git clone --depth 1 https://github.com/chequanghuy/TwinLiteNetPlus.git
%cd {REPO_DIR}

In [ ]:
# torch/torchvision은 위에서 ROCm 빌드로 이미 설치했으니 절대 재설치하지 않는다
# (일반 pip torch가 덮어쓰면 ROCm 가속이 CPU 전용으로 되돌아감) - requirements.txt를
# 그대로 쓰지 않고 나머지 의존성만 개별 설치.
!pip install -q albumentations opencv-python scikit-learn scikit-image scipy pillow tqdm timm matplotlib pyyaml onnx onnxruntime

## 2. Pretrained 가중치

`{BASE_DIR}\pretrained\medium.pth`에 미리 받아둔 게 있으면 그걸 쓰고, 없으면 공식
구글드라이브 폴더에서 gdown으로 시도한다(안 되면 브라우저로 직접 받아서 넣을 것 —
링크: https://drive.google.com/drive/folders/1EqBzUw0b17aEumZmWYrGZmbx_XJqU-vz).

In [ ]:
CONFIG = 'medium'  # 로컬 GPU로 처음 제대로 돌리는 목적이 medium - Colab에서 못 끝낸 것

import os, shutil
os.makedirs(f'{REPO_DIR}/pretrained', exist_ok=True)

local_pretrained = os.path.join(BASE_DIR, 'pretrained', f'{CONFIG}.pth')
target_pretrained = f'{REPO_DIR}/pretrained/{CONFIG}.pth'

if os.path.isfile(local_pretrained):
    shutil.copy(local_pretrained, target_pretrained)
    print('로컬에서 복사:', target_pretrained)
else:
    print(f'{local_pretrained} 없음 - gdown 시도')
    !pip install -q gdown
    !gdown --folder "https://drive.google.com/drive/folders/1EqBzUw0b17aEumZmWYrGZmbx_XJqU-vz" -O {REPO_DIR}/pretrained --remaining-ok

assert os.path.isfile(target_pretrained), f'{target_pretrained} 없음 - 수동으로 받아서 넣을 것'
print('pretrained 준비 완료:', target_pretrained)

## 3. 데이터 준비 (pseudo_dataset_v2, 1068장)

da=bootstrap_v2 모델(134장 순수 사람 라벨 기반, letterbox 버그/pseudo label 오염 없음),
ll=YOLOPv2+skeleton 정제. PROGRESS.md §2.12 참고.

In [ ]:
import os, glob, random, shutil, zipfile

LOCAL_ZIP = os.path.join(BASE_DIR, 'pseudo_dataset_v2.zip')
LOCAL_PSEUDO = os.path.join(BASE_DIR, 'pseudo_dataset_v2')
if not os.path.isdir(LOCAL_PSEUDO):
    assert os.path.isfile(LOCAL_ZIP), f'{LOCAL_ZIP} 없음 - Mac에서 만든 pseudo_dataset_v2.zip을 {BASE_DIR}에 복사해둘 것'
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        zf.extractall(BASE_DIR)
    print('압축 해제 완료:', LOCAL_PSEUDO)

SRC_IMG = os.path.join(LOCAL_PSEUDO, 'images')
SRC_DA = os.path.join(LOCAL_PSEUDO, 'da_masks')
SRC_LL = os.path.join(LOCAL_PSEUDO, 'll_masks')
VAL_RATIO = 0.15
SEED = 42

names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(SRC_IMG, '*.png')))
assert names, f'{SRC_IMG}에 png가 없음'
for n in names:
    for src, ext in [(SRC_DA, '.png'), (SRC_LL, '.png')]:
        assert os.path.isfile(os.path.join(src, n + ext)), f'{n}{ext} 마스크가 {src}에 없음'

random.Random(SEED).shuffle(names)
n_val = max(1, int(len(names) * VAL_RATIO))
val_names, train_names = names[:n_val], names[n_val:]
print(f'전체 {len(names)}장 -> train {len(train_names)} / val {len(val_names)}')

if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
for split, split_names in [('train', train_names), ('val', val_names)]:
    for sub in ['images', 'drivable_area_annotations', 'lane_line_annotations']:
        os.makedirs(os.path.join(DATA_DIR, sub, split), exist_ok=True)
    for n in split_names:
        shutil.copy(os.path.join(SRC_IMG, n + '.png'), os.path.join(DATA_DIR, 'images', split, n + '.png'))
        shutil.copy(os.path.join(SRC_DA, n + '.png'), os.path.join(DATA_DIR, 'drivable_area_annotations', split, n + '.png'))
        shutil.copy(os.path.join(SRC_LL, n + '.png'), os.path.join(DATA_DIR, 'lane_line_annotations', split, n + '.png'))
print('정리 완료:', DATA_DIR)

## 4. 증강 (Colab판과 동일)

글레어/모션블러/바닥반사/가짜선 - 마스크는 그대로, 이미지만 증강(PROGRESS.md §2.10).

In [ ]:
import albumentations as A
import cv2, os, glob, numpy as np, random

N_AUG_PER_IMAGE = 3

glare_aug = A.Compose([
    A.OneOf([
        A.RandomSunFlare(flare_roi=(0, 0, 1, 0.6), src_radius=150, num_flare_circles_range=(3, 8), p=1.0),
        A.RandomBrightnessContrast(brightness_limit=(0.3, 0.6), contrast_limit=(-0.4, -0.1), p=1.0),
    ], p=0.7),
    A.RandomGamma(gamma_limit=(60, 160), p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.5),
])

motion_aug = A.Compose([
    A.MotionBlur(blur_limit=(5, 25), p=1.0),
])

reflect_aug = A.Compose([
    A.RandomSunFlare(flare_roi=(0.0, 0.4, 1.0, 1.0), src_radius=250, num_flare_circles_range=(2, 5), p=0.9),
    A.RandomBrightnessContrast(brightness_limit=(0.1, 0.3), contrast_limit=(-0.2, 0.1), p=0.5),
])

AUG_PIPELINES = [glare_aug, motion_aug, reflect_aug]


def add_distractor_lines(img, n_lines=(2, 5)):
    out = img.copy()
    h, w = out.shape[:2]
    for _ in range(random.randint(*n_lines)):
        x1, y1 = random.randint(0, w - 1), random.randint(int(h * 0.5), h - 1)
        length = random.randint(40, 200)
        angle = random.uniform(-60, 60)
        x2 = int(x1 + length * np.cos(np.radians(angle)))
        y2 = int(y1 - length * np.sin(np.radians(angle)))
        color = random.choice([(200, 200, 200), (230, 230, 210), (60, 60, 60)])
        thickness = random.randint(1, 3)
        overlay = out.copy()
        cv2.line(overlay, (x1, y1), (x2, y2), color, thickness, cv2.LINE_AA)
        alpha = random.uniform(0.25, 0.6)
        out = cv2.addWeighted(overlay, alpha, out, 1 - alpha, 0)
    return out


img_dir = os.path.join(DATA_DIR, 'images', 'train')
da_dir = os.path.join(DATA_DIR, 'drivable_area_annotations', 'train')
ll_dir = os.path.join(DATA_DIR, 'lane_line_annotations', 'train')

n_removed = 0
for d in (img_dir, da_dir, ll_dir):
    for p in glob.glob(os.path.join(d, '*_aug*.png')):
        os.remove(p)
        n_removed += 1
if n_removed:
    print(f'이전 증강 파일 {n_removed}개 정리함 (재실행 대비)')

orig_names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(img_dir, '*.png')))
print(f'원본 train {len(orig_names)}장에 각 {N_AUG_PER_IMAGE}장씩 증강 생성')

for n in orig_names:
    img = cv2.imread(os.path.join(img_dir, n + '.png'))
    for k in range(N_AUG_PER_IMAGE):
        pipeline = random.choice(AUG_PIPELINES)
        aug_img = pipeline(image=img)['image']
        if random.random() < 0.5:
            aug_img = add_distractor_lines(aug_img)
        out_name = f'{n}_aug{k}'
        cv2.imwrite(os.path.join(img_dir, out_name + '.png'), aug_img)
        cv2.imwrite(os.path.join(da_dir, out_name + '.png'), cv2.imread(os.path.join(da_dir, n + '.png'), 0))
        cv2.imwrite(os.path.join(ll_dir, out_name + '.png'), cv2.imread(os.path.join(ll_dir, n + '.png'), 0))

final_count = len(glob.glob(os.path.join(img_dir, '*.png')))
print(f'증강 후 train 총 {final_count}장 (원본 {len(orig_names)}장 -> {final_count}장)')

## 5. 하이퍼파라미터 + 코드 패치 (Colab판과 완전히 동일한 로직)

letterbox 버그 수정(§2.11), ll loss 가중치 2.0배(§2.7), ll 디코더 LR 2.5배(§2.9) —
전부 이미 검증된 패치라 그대로 재사용. 멱등 처리돼 있어 셀 재실행해도 안전함.

In [ ]:
import yaml

with open(f'{REPO_DIR}/hyperparameters/twinlitev2_hyper.yaml') as f:
    hyp = yaml.safe_load(f)

hyp['lr'] = hyp['lr'] * 0.1
hyp['prob_crop'] = 0.0
hyp['ll_loss_weight'] = 2.0
hyp['ll_lr_mult'] = 2.5

FINETUNE_HYP = f'{REPO_DIR}/hyperparameters/finetune_hyper.yaml'
with open(FINETUNE_HYP, 'w') as f:
    yaml.safe_dump(hyp, f)
print('저장:', FINETUNE_HYP)

# --- loss.py 패치: ll_loss_weight ---
loss_path = f'{REPO_DIR}/loss.py'
with open(loss_path) as f:
    loss_src = f.read()

if 'self.ll_weight' in loss_src:
    print('loss.py 이미 패치돼 있음 -> 스킵')
else:
    old_init = '''        self.seg_tver_da = TverskyLoss(mode="multiclass", alpha=alpha1, beta=1-alpha1, gamma=gamma1, from_logits=True)
        self.seg_tver_ll = TverskyLoss(mode="multiclass", alpha=alpha2, beta=1-alpha2, gamma=gamma2, from_logits=True)
        self.seg_focal = FocalLossSeg(mode="multiclass", alpha=alpha3, gamma=gamma3)'''
    new_init = '''        self.seg_tver_da = TverskyLoss(mode="multiclass", alpha=alpha1, beta=1-alpha1, gamma=gamma1, from_logits=True)
        self.seg_tver_ll = TverskyLoss(mode="multiclass", alpha=alpha2, beta=1-alpha2, gamma=gamma2, from_logits=True)
        self.seg_focal = FocalLossSeg(mode="multiclass", alpha=alpha3, gamma=gamma3)
        self.ll_weight = hyp.get("ll_loss_weight", 1.0)'''

    old_sum = '''        tversky_loss,focal_loss=tversky_da_loss+tversky_ll_loss,focal_da_loss+ focal_ll_loss'''
    new_sum = '''        tversky_loss,focal_loss=tversky_da_loss+self.ll_weight*tversky_ll_loss,focal_da_loss+self.ll_weight*focal_ll_loss'''

    assert old_init in loss_src, 'loss.py TotalLoss.__init__ 코드가 예상과 다름'
    assert old_sum in loss_src, 'loss.py TotalLoss.forward 코드가 예상과 다름'
    loss_src = loss_src.replace(old_init, new_init).replace(old_sum, new_sum)
    with open(loss_path, 'w') as f:
        f.write(loss_src)
    print('loss.py 패치 완료')

# --- utils.py 패치: lr_mult 반영 + [:,12:-12] 크롭 제거 ---
utils_path = f'{REPO_DIR}/utils.py'
with open(utils_path) as f:
    utils_src = f.read()

if "lr_mult" in utils_src and "[:,12:-12]" not in utils_src:
    print('utils.py 이미 패치돼 있음 -> 스킵')
else:
    old_sched = '''def poly_lr_scheduler(args, hyp, optimizer, epoch, power=1.5):
    lr = round(hyp['lr'] * (1 - epoch / args.max_epochs) ** power, 8)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    return lr'''
    new_sched = '''def poly_lr_scheduler(args, hyp, optimizer, epoch, power=1.5):
    lr = round(hyp['lr'] * (1 - epoch / args.max_epochs) ** power, 8)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr * param_group.get('lr_mult', 1.0)
    return lr'''
    if old_sched in utils_src:
        utils_src = utils_src.replace(old_sched, new_sched)

    utils_src = utils_src.replace("        da_predict = da_predict[:,12:-12]\n", "")
    utils_src = utils_src.replace("        ll_predict = ll_predict[:,12:-12]\n", "")
    utils_src = utils_src.replace("        predict = predict[:,12:-12]\n", "")

    with open(utils_path, 'w') as f:
        f.write(utils_src)
    print('utils.py 패치 완료')

# --- BDD100K.py 패치: letterbox -> plain resize(640x384) ---
bdd_path = f'{REPO_DIR}/BDD100K.py'
with open(bdd_path) as f:
    bdd_src = f.read()

if 'letterbox(image' not in bdd_src:
    print('BDD100K.py 이미 패치돼 있음 -> 스킵')
else:
    bdd_src = bdd_src.replace('image = letterbox(image, (H_, W_))', 'image = cv2.resize(image, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label, (W_, 360))', 'cv2.resize(label, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label1, (W_, 360))', 'cv2.resize(label1, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label2, (W_, 360))', 'cv2.resize(label2, (W_, H_))')
    with open(bdd_path, 'w') as f:
        f.write(bdd_src)
    print('BDD100K.py 패치 완료 -> letterbox 제거, plain resize(640x384)')

# --- loss.py 크롭 제거 ---
with open(loss_path) as f:
    loss_src2 = f.read()
if '[:,:,12:-12]' not in loss_src2:
    print('loss.py 크롭 이미 제거돼 있음 -> 스킵')
else:
    loss_src2 = loss_src2.replace('out=outputs[:,:,12:-12]', 'out=outputs')
    loss_src2 = loss_src2.replace('out_da,out_ll=out_da[:,:,12:-12],out_ll[:,:,12:-12]', 'out_da,out_ll=out_da,out_ll')
    with open(loss_path, 'w') as f:
        f.write(loss_src2)
    print('loss.py 크롭 제거 완료')

## 6. 파인튜닝 스크립트 작성 (Colab판과 동일, 로컬 백업 경로만 다름)

`best.pth`(da 기준)/`best_ll.pth`(ll 기준) 둘 다 저장, `best_da_miou`/`best_ll_iou`를
checkpoint에 저장해서 재개시 리셋 안 되게 한 버그 수정본(PROGRESS.md §2.12) 포함.

In [ ]:
finetune_script = r'''
import os
import torch
import torch.optim.lr_scheduler
import torch.backends.cudnn as cudnn
import yaml
import math
from copy import deepcopy
from argparse import ArgumentParser

from model.model import TwinLiteNetPlus
from loss import TotalLoss
from utils import train, val, netParams, save_checkpoint, poly_lr_scheduler
import BDD100K

class ModelEMA:
    def __init__(self, model, decay=0.9999, updates=0):
        self.ema = deepcopy(model).eval()
        self.updates = updates
        self.decay = lambda x: decay * (1 - math.exp(-x / 2000))
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            self.updates += 1
            d = self.decay(self.updates)
            msd = model.state_dict()
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point:
                    v *= d
                    v += (1. - d) * msd[k].detach()

def train_net(args, hyp):
    use_ema = args.ema
    cuda_available = torch.cuda.is_available()

    model = TwinLiteNetPlus(args)

    if args.weight and os.path.isfile(args.weight):
        state = torch.load(args.weight, map_location='cpu')
        if isinstance(state, dict) and 'state_dict' in state:
            state = state['state_dict']
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f'[finetune] pretrained 로드: {args.weight}')
        print(f'[finetune]   missing={len(missing)} unexpected={len(unexpected)}')
    else:
        print(f'[finetune] --weight 없음/파일 없음({args.weight}) - 랜덤 초기화로 진행')

    os.makedirs(args.savedir, exist_ok=True)

    trainLoader = torch.utils.data.DataLoader(
        BDD100K.Dataset(hyp, valid=False),
        batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers, pin_memory=True)

    valLoader = torch.utils.data.DataLoader(
        BDD100K.Dataset(hyp, valid=True),
        batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers, pin_memory=True)

    if cuda_available:
        args.onGPU = True
        model = model.cuda()
        cudnn.benchmark = True

    print(f'Total network parameters: {netParams(model)}')

    criteria = TotalLoss(hyp)
    start_epoch = 0
    lr = hyp['lr']
    ll_lr_mult = hyp.get('ll_lr_mult', 1.0)
    ll_param_ids = set()
    ll_params = []
    for name, p in model.named_parameters():
        if '_ll' in name:
            ll_params.append(p)
            ll_param_ids.add(id(p))
    other_params = [p for p in model.parameters() if id(p) not in ll_param_ids]
    print(f'[finetune] ll 디코더 파라미터 {len(ll_params)}개 텐서 (LR x{ll_lr_mult}) / 나머지 {len(other_params)}개 텐서')
    optimizer = torch.optim.AdamW([
        {'params': other_params, 'lr': lr, 'lr_mult': 1.0},
        {'params': ll_params, 'lr': lr * ll_lr_mult, 'lr_mult': ll_lr_mult},
    ], betas=(hyp['momentum'], 0.999), eps=hyp['eps'], weight_decay=hyp['weight_decay'])

    ema = ModelEMA(model) if use_ema else None

    best_da_miou = -1.0
    best_ll_iou = -1.0
    if args.resume and os.path.isfile(args.resume):
        if args.resume.endswith('.tar'):
            print(f"=> Loading checkpoint '{args.resume}'")
            checkpoint = torch.load(args.resume)
            start_epoch = checkpoint['epoch']
            model.load_state_dict(checkpoint['state_dict'])
            if use_ema:
                ema.ema.load_state_dict(checkpoint['ema_state_dict'])
                ema.updates = checkpoint['updates']
            optimizer.load_state_dict(checkpoint['optimizer'])
            best_da_miou = checkpoint.get('best_da_miou', -1.0)
            best_ll_iou = checkpoint.get('best_ll_iou', -1.0)
            print(f"=> Loaded checkpoint (epoch {checkpoint['epoch']}, best_da_miou={best_da_miou:.3f}, best_ll_iou={best_ll_iou:.3f})")
        else:
            print(f"=> No valid checkpoint found at '{args.resume}'")

    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(start_epoch, args.max_epochs):
        model_file_name = os.path.join(args.savedir, f'model_{epoch}.pth')
        poly_lr_scheduler(args, hyp, optimizer, epoch)
        lr = optimizer.param_groups[0]['lr']
        ll_lr = optimizer.param_groups[1]['lr'] if len(optimizer.param_groups) > 1 else lr
        print(f'Learning rate: {lr} (ll decoder: {ll_lr})')

        model.train()
        ema = train(args, trainLoader, model, criteria, optimizer, epoch, scaler, args.verbose, ema if use_ema else None)

        model.eval()
        da_segment_results, ll_segment_results = val(valLoader, ema.ema if use_ema else model, args=args)

        print(f'Driving Area Segment: mIOU({da_segment_results[2]:.3f})')
        print(f'Lane Line Segment: Acc({ll_segment_results[0]:.3f}) IOU({ll_segment_results[1]:.3f})')

        torch.save(ema.ema.state_dict(), model_file_name) if use_ema else torch.save(model.state_dict(), model_file_name)
        if da_segment_results[2] > best_da_miou:
            best_da_miou = da_segment_results[2]
            best_path = os.path.join(args.savedir, 'best.pth')
            torch.save(ema.ema.state_dict() if use_ema else model.state_dict(), best_path)
            print(f'[finetune] 새 best(da) 저장: {best_path} (da mIoU={best_da_miou:.3f})')
        if ll_segment_results[1] > best_ll_iou:
            best_ll_iou = ll_segment_results[1]
            best_ll_path = os.path.join(args.savedir, 'best_ll.pth')
            torch.save(ema.ema.state_dict() if use_ema else model.state_dict(), best_ll_path)
            print(f'[finetune] 새 best(ll) 저장: {best_ll_path} (ll IOU={best_ll_iou:.3f})')

        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'ema_state_dict': ema.ema.state_dict() if use_ema else None,
            'updates': ema.updates if use_ema else None,
            'optimizer': optimizer.state_dict(),
            'lr': lr,
            'best_da_miou': best_da_miou,
            'best_ll_iou': best_ll_iou,
        }, os.path.join(args.savedir, 'checkpoint.pth.tar'))

        if args.drive_backup_dir:
            import shutil as _shutil
            os.makedirs(args.drive_backup_dir, exist_ok=True)
            for _fn in ('checkpoint.pth.tar', 'best.pth', 'best_ll.pth'):
                _src = os.path.join(args.savedir, _fn)
                if os.path.isfile(_src):
                    _shutil.copy(_src, os.path.join(args.drive_backup_dir, _fn))
            print(f'[finetune] epoch {epoch} 체크포인트 백업 완료: {args.drive_backup_dir}')

if __name__ == '__main__':
    parser = ArgumentParser()
    parser.add_argument('--max_epochs', type=int, default=100)
    parser.add_argument('--num_workers', type=int, default=4)
    parser.add_argument('--batch_size', type=int, default=8)
    parser.add_argument('--savedir', default='./finetune_out')
    parser.add_argument('--hyp', type=str, default='./hyperparameters/finetune_hyper.yaml')
    parser.add_argument('--resume', type=str, default='')
    parser.add_argument('--weight', type=str, default='')
    parser.add_argument('--config', default='small')
    parser.add_argument('--verbose', action='store_true')
    parser.add_argument('--ema', action='store_true')
    parser.add_argument('--drive_backup_dir', type=str, default='')
    args = parser.parse_args()

    with open(args.hyp, errors='ignore') as f:
        hyp = yaml.safe_load(f)

    train_net(args, hyp.copy())
'''

with open(f'{REPO_DIR}/finetune.py', 'w') as f:
    f.write(finetune_script)
print('작성 완료:', f'{REPO_DIR}/finetune.py')

## 7. 학습 실행

9070 XT(16GB VRAM)면 medium config에 `BATCH_SIZE`를 Colab T4보다 올려도 될 것 —
일단 8로 시작하고 VRAM 여유 있으면(작업관리자/`rocm-smi`로 확인) 16~24로 올려볼 것.
Colab과 달리 세션 만료 걱정은 없지만, 정전/재부팅 대비로 `--drive_backup_dir`을
다른 드라이브(예: OneDrive 폴더나 외장 SSD)로 지정해두면 안전함 — 굳이 안 써도 됨.

In [ ]:
%cd {REPO_DIR}
MAX_EPOCHS = 40
BATCH_SIZE = 8
SAVEDIR = os.path.join(BASE_DIR, 'finetune_out_local_medium')
BACKUP_DIR = ''  # 필요하면 다른 드라이브 경로로, 예: r'D:\backup\finetune_out_local_medium'

_resume_ckpt = os.path.join(BACKUP_DIR, 'checkpoint.pth.tar') if BACKUP_DIR else ''
RESUME_FLAG = f'--resume {_resume_ckpt}' if _resume_ckpt and os.path.isfile(_resume_ckpt) else ''
if RESUME_FLAG:
    print(f'[재개] 백업 체크포인트 발견 -> 이어서 학습: {_resume_ckpt}')
else:
    print('[신규] 처음부터 학습')

train_cmd = (
    f'python finetune.py --config {CONFIG} --weight pretrained/{CONFIG}.pth '
    f'--hyp hyperparameters/finetune_hyper.yaml --max_epochs {MAX_EPOCHS} '
    f'--batch_size {BATCH_SIZE} --savedir "{SAVEDIR}" --drive_backup_dir "{BACKUP_DIR}" '
    f'{RESUME_FLAG} --ema --verbose'
)
print(train_cmd)
get_ipython().system(train_cmd)

## 8. ONNX로 내보내기

`track_drive`의 `TwinLiteNetEngine`과 그대로 호환(`DL_INPUT_NAME='images'`,
`DL_OUTPUT_NAMES=('da','ll')`). 실차 반영 시 `perception/dl_lane.py`의
`DL_INPUT_H = 360` -> **384**로 바꿔야 함(다른 노트북들과 동일).

In [ ]:
import torch, os
from model.model import TwinLiteNetPlus
from argparse import Namespace

WEIGHT_PATH = os.path.join(SAVEDIR, 'best.pth')  # ll 기준 best는 best_ll.pth
ONNX_OUT = os.path.join(BASE_DIR, f'twinlitenetplus_{CONFIG}_finetuned_local.onnx')

model = TwinLiteNetPlus(Namespace(config=CONFIG))
model.load_state_dict(torch.load(WEIGHT_PATH, map_location='cpu'))
model.eval()

dummy = torch.zeros(1, 3, 384, 640)
torch.onnx.export(
    model, dummy, ONNX_OUT,
    input_names=['images'], output_names=['da', 'll'],
    dynamic_axes={'images': {0: 'batch'}, 'da': {0: 'batch'}, 'll': {0: 'batch'}},
    opset_version=12,
)
print('내보내기 완료:', ONNX_OUT)

In [ ]:
# onnx가 pytorch와 같은 결과를 내는지 검증
import onnxruntime as ort, numpy as np

sess = ort.InferenceSession(ONNX_OUT, providers=['CPUExecutionProvider'])
x = dummy.numpy()
onnx_da, onnx_ll = sess.run(['da', 'll'], {'images': x})

with torch.no_grad():
    torch_da, torch_ll = model(dummy)

da_diff = np.abs(onnx_da - torch_da.numpy()).max()
ll_diff = np.abs(onnx_ll - torch_ll.numpy()).max()
print(f'da 최대 오차: {da_diff:.6f} | ll 최대 오차: {ll_diff:.6f} (1e-4 이하면 정상)')
assert da_diff < 1e-3 and ll_diff < 1e-3, 'ONNX 변환 결과가 PyTorch와 다름'
print('검증 통과')